In [1]:
import os
import codecs
import sys
import json
from astropy.io import fits
import numpy as np
import math
from astropy import units as u
from astropy.wcs import WCS
from scipy.ndimage import binary_dilation, generic_filter
import pandas as pd
import matplotlib.pyplot as plt
import pkg_resources
import re
from astropy import constants as ac
from tqdm import tqdm
from scipy import interpolate
import sympy as sp
import seaborn as sns
from pint import UnitRegistry
from astropy.table import Table, vstack
import subprocess

In [2]:
def run_command(command, cwd=None):
    """Run a command and print its output."""
    print(f"Running command: {command}")
    result = subprocess.run(command, shell=True, cwd=cwd, text=True, capture_output=True)
    if result.returncode != 0:
        print(f"Error running command: {result.stderr}")
    else:
        print(result.stdout)

def find_starlink_setup(base_paths, filename="profile"):
    """Recursively search for a file named `filename` starting from `base_paths`."""
    for base_path in base_paths:
        for root, dirs, files in os.walk(base_path):
            if filename in files:
                return os.path.join(root, filename)
    return None

In [3]:
folder_path='/Users/kyriakos.trakakis/Downloads/CHIMPS_COHRS/13CO'
fgn_files={}
for filename in os.listdir(folder_path):
        if filename.endswith(".fits"):
            # Extract the identifier (e.g., 018 from FGN_018_12CO_23arcsec.fits or CHIMPS_12CO_G018.fits)
            if filename.endswith("23arcsec_aligned_SNR.fits"):
                identifier = filename.split("_")[1]  # Extract '018' from 'FGN_018_12CO_23arcsec.fits'
                fgn_files[identifier] = os.path.join(folder_path, filename)  # Include folder path in file path

# Sort the dictionary by keys in ascending order
fgn_files_sorted = dict(sorted(fgn_files.items(), key=lambda item: int(item[0])))

In [37]:
for identifier in fgn_files_sorted:
    map_sdf = fgn_files_sorted[identifier].replace(".fits", ".sdf")
    run_command(f"/usr/local/star-2023A/bin/convert/fits2ndf {fgn_files_sorted[identifier]} {map_sdf}")


Running command: /usr/local/star-2023A/bin/convert/fits2ndf /Users/kyriakos.trakakis/Downloads/CHIMPS_COHRS/13CO/FGN_018_13CO_23arcsec_aligned_SNR.fits /Users/kyriakos.trakakis/Downloads/CHIMPS_COHRS/13CO/FGN_018_13CO_23arcsec_aligned_SNR.sdf
1 file selected.

Running command: /usr/local/star-2023A/bin/convert/fits2ndf /Users/kyriakos.trakakis/Downloads/CHIMPS_COHRS/13CO/FGN_020_13CO_23arcsec_aligned_SNR.fits /Users/kyriakos.trakakis/Downloads/CHIMPS_COHRS/13CO/FGN_020_13CO_23arcsec_aligned_SNR.sdf
1 file selected.

Running command: /usr/local/star-2023A/bin/convert/fits2ndf /Users/kyriakos.trakakis/Downloads/CHIMPS_COHRS/13CO/FGN_022_13CO_23arcsec_aligned_SNR.fits /Users/kyriakos.trakakis/Downloads/CHIMPS_COHRS/13CO/FGN_022_13CO_23arcsec_aligned_SNR.sdf
1 file selected.

Running command: /usr/local/star-2023A/bin/convert/fits2ndf /Users/kyriakos.trakakis/Downloads/CHIMPS_COHRS/13CO/FGN_024_13CO_23arcsec_aligned_SNR.fits /Users/kyriakos.trakakis/Downloads/CHIMPS_COHRS/13CO/FGN_024_13CO

In [42]:
folder_path='/Users/kyriakos.trakakis/Downloads/CHIMPS_COHRS/13CO'
fgn_sdf_files={}
for filename in os.listdir(folder_path):
        if filename.endswith(".sdf"):
            # Extract the identifier (e.g., 018 from FGN_018_12CO_23arcsec.fits or CHIMPS_12CO_G018.fits)
            if filename.endswith("23arcsec_aligned_SNR.sdf"):
                identifier = filename.split("_")[1]  # Extract '018' from 'FGN_018_12CO_23arcsec.fits'
                fgn_sdf_files[identifier] = os.path.join(folder_path, filename)  # Include folder path in file path

# Sort the dictionary by keys in ascending order
fgn_sdf_files_sorted = dict(sorted(fgn_sdf_files.items(), key=lambda item: int(item[0])))

In [48]:
# Define folder path and mosaic output file
sdf_folder = '/Users/kyriakos.trakakis/Downloads/CHIMPS_COHRS/13CO'
output_mosaic = os.path.join(sdf_folder, 'FGN_017_047_13CO_23arcsec_aligned_SNR.sdf')

# Collect all .sdf files, sort them by the numerical identifier
sdf_files = sorted(
    [os.path.join(sdf_folder, f) for f in os.listdir(sdf_folder) if f.endswith('.sdf')],
    key=lambda x: int(os.path.basename(x).split('_')[1])  # Extract identifier for sorting
)

# Prepare the command with all files as arguments
sdf_files_str = ' '.join(sdf_files)
command = f"/usr/local/star-2023A/bin/kappa/wcsmosaic {sdf_files_str} {output_mosaic}"

try:
    # Execute the command as a single string
    subprocess.run(command, shell=True, check=True)
    print(f"Mosaic created successfully: {output_mosaic}")
except subprocess.CalledProcessError as e:
    print(f"Error running WCSMOSAIC: {e}")


!! SUBPAR: Attempt to use 'positional' parameter value
!     (/Users/kyriakos.trakakis/Downloads/CHIMPS_COHRS/13CO/FGN_026_13CO_23arcs
!     ec_aligned_SNR.sdf) in an unallocated position
!  Application exit status SUBPAR__PNOTAL, command line parameter position
!     used is not allocated
!  /Users/kyriakos.trakakis/Downloads/CHIMPS_COHRS/13CO/FGN_017_027_13CO_23arcs
!     ec_aligned_SNR.sdf
!     /Users/kyriakos.trakakis/Downloads/CHIMPS_COHRS/13CO/FGN_018_13CO_23arcse
!     c_aligned_SNR.sdf /Users/kyr...
Mosaic created successfully: /Users/kyriakos.trakakis/Downloads/CHIMPS_COHRS/13CO/FGN_017_047_13CO_23arcsec_aligned_SNR.sdf
